# Smart LMS v4 - Engagement Detection Model Training
## Temporal Transformer + BiLSTM-GRU + XGBoost + Stacking Ensemble

**GPU Runtime Required:** T4 GPU (30 hrs/week budget)

### Datasets needed:
1. `smartlms-openface` — Pre-extracted OpenFace features (CSVs + labels)
2. `smartlms-vit-embeddings` — ViT face embeddings (Week 2, optional)

### Training Plan:
- **Week 1:** Transformer + BiLSTM-GRU + XGBoost CV (~15 hrs)
- **Week 2:** ViT embedding extraction + ViT classifier (~12 hrs)
- **Week 3:** Stacking ensemble + ablation study (~8 hrs)

In [ ]:
# ── 0. Setup & Install Dependencies ──
!pip install -q xgboost optuna shap transformers

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ── 1. Copy training scripts to working directory ──
import os, shutil, sys

# The training scripts should be uploaded as part of the smartlms-openface dataset
# or copy them in this cell
WORK_DIR = "/kaggle/working"
os.makedirs(f"{WORK_DIR}/app/ml", exist_ok=True)

# Check if scripts exist in dataset
script_sources = [
    "/kaggle/input/smartlms-openface/train_model_v2.py",
    "/kaggle/input/smartlms-openface/train_model_v3.py",
    "/kaggle/input/smartlms-openface/train_model_v4.py",
]
for src in script_sources:
    if os.path.exists(src):
        dst = f"{WORK_DIR}/app/ml/{os.path.basename(src)}"
        shutil.copy(src, dst)
        print(f"Copied: {src} -> {dst}")

# Add to path
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── 2. Verify Data ──
import glob

OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"
LABELS_DIR = "/kaggle/input/smartlms-openface/Labels"

# Check files
csvs = glob.glob(f"{OPENFACE_DIR}/*.csv")
print(f"OpenFace CSVs: {len(csvs)}")

label_files = glob.glob(f"{LABELS_DIR}/*.csv")
print(f"Label files: {[os.path.basename(f) for f in label_files]}")

# Quick peek at a CSV
import pandas as pd
if csvs:
    sample = pd.read_csv(csvs[0])
    print(f"\nSample CSV shape: {sample.shape}")
    print(f"Columns: {list(sample.columns[:10])}...")

## Week 1: Temporal Transformer + BiLSTM-GRU + XGBoost CV
Expected runtime: ~15 hours on T4 GPU

In [ ]:
# ── 3a. Train Temporal Transformer (all dimensions) ──
# ~40 min per dimension × 4 = ~2.5 hrs
!python app/ml/train_model_v4.py --mode transformer --seq_len 60 --transformer_epochs 120

In [ ]:
# ── 3b. Train BiLSTM-GRU Hybrid (all dimensions) ──
# ~30 min per dimension × 4 = ~2 hrs
!python app/ml/train_model_v4.py --mode bilstm_v4 --seq_len 60 --bilstm_epochs 100

In [ ]:
# ── 3c. XGBoost with 5-Fold Cross-Validation (all dimensions) ──
# ~30 min per dimension × 4 = ~2 hrs
!python app/ml/train_model_v4.py --mode xgboost_cv --n_folds 5 --n_trials 30

In [ ]:
# ── 3d. Stacking Ensemble (all dimensions) ──
# Uses saved probabilities from 3a/3b/3c
!python app/ml/train_model_v4.py --mode stacking

In [ ]:
# ── 4. Review Results ──
import json, glob

results_files = sorted(glob.glob("/kaggle/working/experiment_results/experiment_v4_*.json"))
for rf in results_files:
    print(f"\n{'='*60}")
    print(f"File: {os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, res in data.get('results', {}).items():
        f1m = res.get('test_f1_macro') or res.get('cv_f1_macro_mean') or res.get('stacking_f1_macro', 0)
        std = res.get('cv_f1_macro_std') or res.get('stacking_cv_std', '')
        std_str = f" ± {std:.4f}" if std else ""
        print(f"  {key}: F1m = {f1m:.4f}{std_str}")

## Week 2: ViT Face Embedding Extraction + Classifier

**Prerequisites:** Upload raw DAiSEE videos as a Kaggle dataset

Expected runtime: ~8 hrs for embedding extraction + ~4 hrs for training

In [ ]:
# ── 5a. Extract ViT Face Embeddings ──
# Only run this if you have uploaded DAiSEE raw videos
# ~8 hrs for full dataset
VIDEO_DIR = "/kaggle/input/daisee-videos/DAiSEE/DataSet"  # Adjust path
if os.path.exists(VIDEO_DIR):
    !python app/ml/extract_face_embeddings.py \
        --video_dir {VIDEO_DIR} \
        --output_dir /kaggle/working/vit_embeddings \
        --sample_fps 2.0 \
        --batch_size 32 \
        --resume
else:
    print(f"Video directory not found: {VIDEO_DIR}")
    print("Upload DAiSEE raw videos as a Kaggle dataset first.")

In [ ]:
# ── 5b. Train ViT Embedding Classifier ──
VIT_DIR = "/kaggle/working/vit_embeddings"
if os.path.exists(VIT_DIR) and len(glob.glob(f"{VIT_DIR}/*.npy")) > 100:
    !python app/ml/train_model_v4.py --mode vit_train
else:
    print("ViT embeddings not yet extracted. Run cell 5a first.")

## Week 3: Final Stacking + Full Pipeline

In [ ]:
# ── 6. Full Pipeline: All models + stacking ──
# Only run if ViT embeddings are available, otherwise use all_openface
VIT_DIR = "/kaggle/working/vit_embeddings"
mode = "full" if os.path.exists(VIT_DIR) else "all_openface"
print(f"Running mode: {mode}")
!python app/ml/train_model_v4.py --mode {mode} --n_folds 5 --n_trials 30

In [ ]:
# ── 7. Download trained models ──
import shutil

# Package everything for download
output_dirs = [
    "/kaggle/working/trained_models",
    "/kaggle/working/experiment_results",
]

for d in output_dirs:
    if os.path.exists(d):
        n = len(os.listdir(d))
        print(f"{d}: {n} files")

# Create zip for easy download
shutil.make_archive("/kaggle/working/v4_models", 'zip', "/kaggle/working/trained_models")
shutil.make_archive("/kaggle/working/v4_results", 'zip', "/kaggle/working/experiment_results")
print("\nZip files created for download:")
for f in glob.glob("/kaggle/working/*.zip"):
    print(f"  {f} ({os.path.getsize(f) / 1e6:.1f} MB)")

In [ ]:
# ── 8. Compare v3 vs v4 Results ──
print("\n" + "="*80)
print("V3 BASELINE vs V4 COMPARISON")
print("="*80)

v3_baseline = {
    "XGBoost v3": {"boredom": 0.575, "engagement": 0.616, "confusion": 0.551, "frustration": 0.538},
    "BiLSTM v3":  {"boredom": 0.532, "engagement": 0.566, "confusion": 0.525, "frustration": 0.526},
    "Ensemble v3": {"boredom": 0.573, "engagement": 0.601, "confusion": 0.555, "frustration": 0.521},
}

# Load v4 results
v4_results = {}
for rf in sorted(glob.glob("/kaggle/working/experiment_results/experiment_v4_*.json")):
    with open(rf) as f:
        data = json.load(f)
    for key, res in data.get('results', {}).items():
        f1m = res.get('test_f1_macro') or res.get('cv_f1_macro_mean') or res.get('stacking_f1_macro', 0)
        v4_results[key] = f1m

print(f"\n{'Model':<30} {'Boredom':>10} {'Engage':>10} {'Confuse':>10} {'Frustr':>10} {'Avg':>10}")
print("-"*80)

for model_name, dims in v3_baseline.items():
    values = list(dims.values())
    avg = sum(values) / len(values)
    print(f"{model_name:<30} {values[0]:10.4f} {values[1]:10.4f} {values[2]:10.4f} {values[3]:10.4f} {avg:10.4f}")

print("-"*80)

# Group v4 by model type
from collections import defaultdict
v4_by_model = defaultdict(dict)
for key, f1m in v4_results.items():
    parts = key.rsplit('_', 1)
    if len(parts) == 2:
        model_type, dim = parts
        v4_by_model[model_type][dim] = f1m

for model_name, dims in v4_by_model.items():
    values = [dims.get(d, 0) for d in ["boredom", "engagement", "confusion", "frustration"]]
    avg = sum(values) / max(len(values), 1)
    print(f"{model_name + ' v4':<30} {values[0]:10.4f} {values[1]:10.4f} {values[2]:10.4f} {values[3]:10.4f} {avg:10.4f}")

print("\n\u2713 If stacking v4 avg > 0.60, the improvements are significant!")